# PyTorch Fundamentals and Training Loop

Practice tensor construction and reshaping, NumPy interchange, autograd, Dataset/DataLoader, nn.Module, correct train/eval modes, and loss aggregation.

- **Study time:** 45-60 minutes
- **Prerequisites:** NumPy shapes, derivatives, and Python classes
- **Mode:** `optional`
- **Data policy:** no downloads; seeded synthetic tensors only; any checkpoint path resolves outside the vault
- **Provenance:** consolidated from the legacy PyTorch introduction and training-loop drills

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, subtle API behavior, or configuration side effects; obvious Python syntax is left uncommented.


In [ ]:
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from datacoding.config import external_path

seed = 51
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)  # Align all three RNGs for this executable example.
# Select one device reused by model and batches.
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print("Environment | torch version", torch.__version__)
print("Environment | selected device", device)

## 1. Tensor construction, shape operations, and combining

`torch.cat` joins an existing dimension; `torch.stack` creates a new one. As in NumPy, broadcasting aligns dimensions from the right.


In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
bias = torch.tensor([1.0, 2.0, 3.0, 4.0])  # Shape (4,) aligns with X's trailing feature axis.
shifted = X + bias
expanded = X.unsqueeze(0)  # Insert batch dimension: (1, 3, 4).
permuted = expanded.permute(
    0, 2, 1
)  # Reorder via strides: (1, 4, 3); result may be non-contiguous.
concatenated = torch.cat([X, X], dim=0)  # Extend an existing dimension: (6, 4).
stacked = torch.stack([X, X], dim=0)  # Create a new dimension: (2, 3, 4).

print("Tensor | X shape/dtype/device", (tuple(X.shape), X.dtype, X.device))
print("Broadcast | X + bias shape", tuple(shifted.shape))
print("Broadcast | shifted values", shifted)
print("Shape | unsqueeze then permute", (tuple(expanded.shape), tuple(permuted.shape)))
print(
    "Combine | cat existing dim versus stack new dim",
    (tuple(concatenated.shape), tuple(stacked.shape)),
)

## 2. NumPy interchange and memory ownership

`torch.from_numpy` shares CPU memory with its NumPy input, while `torch.tensor` copies. Before converting a tensor produced by a model, leave autograd and move to CPU with `detach().cpu().numpy()`.


In [ ]:
numpy_source = np.arange(6, dtype=np.float32).reshape(2, 3)
shared_tensor = torch.from_numpy(numpy_source)  # Shares CPU storage with numpy_source.
copied_tensor = torch.tensor(numpy_source)  # Owns independent storage.
numpy_source[0, 0] = -1.0  # Mutation appears only through the shared tensor.
detached_numpy = shifted.detach().cpu().numpy()  # Leave autograd, then ensure CPU-backed memory.

print("NumPy boundary | source after mutation", numpy_source)
print(
    "NumPy boundary | from_numpy shares, tensor copies",
    (shared_tensor[0, 0].item(), copied_tensor[0, 0].item()),
)
print("NumPy boundary | detached CPU array shape", detached_numpy.shape)

## 3. Autograd and gradient accumulation


In [ ]:
weight = torch.tensor(2.0, requires_grad=True)
loss = (weight * 3.0 - 7.0) ** 2
loss.backward()  # Populate weight.grad through the recorded computation graph.
first_gradient = weight.grad.item()
second_loss = (weight * 3.0 - 7.0) ** 2
second_loss.backward()  # Gradients add to existing .grad by default.
accumulated_gradient = weight.grad.item()
weight.grad.zero_()  # Clear in place before a future optimization step.

print("Autograd | scalar loss", loss.item())
print("Autograd | first gradient", first_gradient)
print("Autograd | accumulated after second backward", accumulated_gradient)
print("Autograd | gradient after zero", weight.grad.item())

## 4. Dataset and DataLoader contract


In [ ]:
n_rows, n_features = 1_200, 5
features = torch.randn(n_rows, n_features)
true_weight = torch.tensor([1.5, -2.0, 0.5, 3.0, -1.0]).reshape(
    -1, 1
)  # Column shape keeps targets (n, 1).
targets = features @ true_weight + 0.4 + 0.2 * torch.randn(n_rows, 1)

train_dataset = TensorDataset(features[:900], targets[:900])
validation_dataset = TensorDataset(features[900:], targets[900:])
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True
)  # Reshuffle training order each epoch.
validation_loader = DataLoader(
    validation_dataset, batch_size=128, shuffle=False
)  # Stable evaluation order.
sample_X, sample_y = next(iter(train_loader))

print(
    "DataLoader | feature and target batch shapes", (tuple(sample_X.shape), tuple(sample_y.shape))
)

## 5. Model and canonical loops


In [ ]:
class TinyRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, X):
        return self.network(X)  # Preserve the leading batch dimension: (batch, 1).


def train_epoch(model, loader, optimizer, loss_fn):
    model.train()  # Enable dropout/batch-norm training behavior; gradient tracking is separate.
    total_loss = 0.0
    total_examples = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(
            set_to_none=True
        )  # Clear prior gradients; None can avoid an eager zero-fill.
        prediction = model(X_batch)
        loss = loss_fn(prediction, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)  # Convert batch mean to a sample-weighted sum.
        total_examples += len(X_batch)
    return total_loss / total_examples


def evaluate(model, loader, loss_fn):
    model.eval()  # Use inference behavior; no_grad below separately disables graph construction.
    total_loss = 0.0
    total_examples = 0
    with torch.no_grad():  # Avoid building graphs that evaluation will never backpropagate through.
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            loss = loss_fn(model(X_batch), y_batch)
            total_loss += loss.item() * len(
                X_batch
            )  # Keep the epoch mean correct for a short final batch.
            total_examples += len(X_batch)
    return total_loss / total_examples

In [ ]:
model = TinyRegressor(n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()  # Default batch-mean reduction is reweighted during epoch aggregation.

history = []
for epoch in range(1, 7):
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
    validation_loss = evaluate(model, validation_loader, loss_fn)
    history.append((train_loss, validation_loss))
    print(
        f"Training | epoch={epoch:02d} train_mse={train_loss:.4f} validation_mse={validation_loss:.4f}"
    )

print("Training | first and final loss pairs", (history[0], history[-1]))

## 6. Save learned state outside the vault


In [ ]:
# Resolve/create an artifact path outside the vault.
checkpoint_path = external_path("models", "tiny_regressor_state.pt", create_parent=True)
# Store learned tensors, not the Python model object.
torch.save(model.state_dict(), checkpoint_path)

restored = TinyRegressor(n_features).to(device)
restored.load_state_dict(
    torch.load(checkpoint_path, map_location=device, weights_only=True)
)  # Restore safely across devices.
restored_loss = evaluate(restored, validation_loader, loss_fn)

assert concatenated.shape == (6, 4)
assert stacked.shape == (2, 3, 4)
assert accumulated_gradient == 2 * first_gradient
print("Checkpoint | external path", checkpoint_path)
print("Checkpoint | restored validation MSE", restored_loss)
print(
    "Training checks | status", "model modes, no-grad evaluation, and external state_dict verified"
)